In [3]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

df = pd.read_excel("Customer_Sentiment_Feedback_Uncleaned.xlsx", sheet_name="Customer_Feedback")
print("Before:", df.shape)
df.head()

Before: (4000, 14)


,Customer_ID,Order_ID,Age,Gender,Product_Name,Product_Category,Rating,Review_Date,Review_Text,Sentiment,Email,Phone,Purchase_Amount,City
0,CUST1000,ORD500000,61,Male,Wireless Earbuds,Electronics,6,2023-08-17,Great quality and fast delivery. Highly recomm...,Positive,ravisharma605@hotmail.com,6013389083,14376.19,Kochi
1,CUST1001,ORD500001,62,Female,Cookbook,Bookss,3,2023-10-12,"It works as described, no complaints no praise.",Neutral,aruniyer349@outlook.com,7351161559,5436.51,Chennai
2,CUST1002,ORD500002,41,Male,Lipstick,Beauty,2,2023-03-22,"Terrible experience, would not recommend.",Negative,lakshmimenon722@gmail.com,6341316475,3419.55,Kolkata
3,CUST1003,ORD500003,64,Male,Running Shoes,Clothing,5,2024-09-15,"Absolutely love this product, works perfectly!",Positive,meenadas168@hotmail.com,9483503056,5500.27,Bangalore
4,NaN,ORD500004,34,Male,Mystery Novel,Books,3,2024-02-10,"Decent for the price, could be better.",Neutral,manojiyer253@outlook.com,9965328,17248.84,Delhi


In [4]:
# 1) Customer_ID - missing values
df["Customer_ID"] = df["Customer_ID"].fillna("UNKNOWN")
print(df["Customer_ID"])

0       CUST1000
1       CUST1001
2       CUST1002
3       CUST1003
4        UNKNOWN
          ...   
3995    CUST4995
3996    CUST4996
3997    CUST4997
3998    CUST4998
3999    CUST4999
Name: Customer_ID, Length: 4000, dtype: str


In [4]:
# 2) Order_ID - duplicates
df = df.drop_duplicates(subset="Order_ID", keep="first")
print(df)

     Customer_ID   Order_ID Age  Gender          Product_Name  \
0       CUST1000  ORD500000  61    Male      Wireless Earbuds   
1       CUST1001  ORD500001  62  Female              Cookbook   
2       CUST1002  ORD500002  41    Male              Lipstick   
3       CUST1003  ORD500003  64    Male       Running Shoes     
4        UNKNOWN  ORD500004  34    Male         Mystery Novel   
...          ...        ...  ..     ...                   ...   
3994    CUST4994  ORD503994  38    MALE           Face   Wash   
3995    CUST4995  ORD503995  26  Female  Self-Help Bestseller   
3996    CUST4996  ORD503996  40    Male        Cotton T-Shirt   
3998    CUST4998  ORD503998  70  Female           LED Monitor   
3999    CUST4999  ORD503999  29  Female         Running Shoes   

     Product_Category  Rating Review_Date  \
0         Electronics       6  2023-08-17   
1              Bookss       3  2023-10-12   
2              Beauty       2  2023-03-22   
3            Clothing       5  2024-09-

In [5]:
#age

df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Age"] = df["Age"].fillna(df["Age"].median()).astype(int)
print(df["Age"])

0       61
1       62
2       41
3       64
4       34
        ..
3995    26
3996    40
3997    27
3998    70
3999    29
Name: Age, Length: 4000, dtype: int64


In [6]:
#gender 

df["Gender"] = (df["Gender"].str.strip().str.lower()
                .map({"male": "Male", "m": "Male", "female": "Female", "f": "Female"}))
print(df["Gender"])

0         Male
1       Female
2         Male
3         Male
4         Male
         ...  
3995    Female
3996      Male
3997    Female
3998    Female
3999    Female
Name: Gender, Length: 4000, dtype: str


In [7]:
#product name extra space
df["Product_Name"] = df["Product_Name"].str.strip().str.replace(r"\s+", " ", regex=True)
print(df["Product_Name"])

0           Wireless Earbuds
1                   Cookbook
2                   Lipstick
3              Running Shoes
4              Mystery Novel
                ...         
3995    Self-Help Bestseller
3996          Cotton T-Shirt
3997              Hair Serum
3998             LED Monitor
3999           Running Shoes
Name: Product_Name, Length: 4000, dtype: str


In [8]:
#product category spelling mistakes
valid_cats = ["Electronics", "Books", "Clothing", "Beauty", "Home & Kitchen"]
def fix_cat(x):
    m = get_close_matches(str(x).strip().title(), valid_cats, n=1, cutoff=0.6)
    return m[0] if m else np.nan
df["Product_Category"] = df["Product_Category"].apply(fix_cat)
print(df["Product_Category"])

0       Electronics
1             Books
2            Beauty
3          Clothing
4             Books
           ...     
3995          Books
3996       Clothing
3997         Beauty
3998    Electronics
3999       Clothing
Name: Product_Category, Length: 4000, dtype: str


In [9]:
#rating
df["Rating"] = df["Rating"].where(df["Rating"].between(1, 5))
print(df["Rating"])

0       NaN
1       3.0
2       2.0
3       5.0
4       3.0
       ... 
3995    4.0
3996    5.0
3997    4.0
3998    4.0
3999    4.0
Name: Rating, Length: 4000, dtype: float64


In [10]:
#review date
def parse_date(s):
    s = str(s).strip()
    for fmt in ("%Y-%m-%d", "%Y/%m/%d",
                "%m/%d/%Y",
                "%d-%m-%Y", "%d.%m.%Y",
                "%d %b %Y", "%B %d, %Y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except ValueError:
            pass
    return pd.NaT
df["Review_Date"] = df["Review_Date"].apply(parse_date).dt.strftime("%Y-%m-%d")
print(df["Review_Date"])

0       2023-08-17
1       2023-10-12
2       2023-03-22
3       2024-09-15
4       2024-02-10
           ...    
3995    2025-01-03
3996    2023-02-20
3997    2024-06-01
3998    2025-09-15
3999    2023-10-31
Name: Review_Date, Length: 4000, dtype: str


In [11]:
#Review_Text: HTML tag
df["Review_Text"] = df["Review_Text"].str.replace(r"<[^>]+>", "", regex=True).str.strip()
print(df["Review_Text"])

0       Great quality and fast delivery. Highly recomm...
1         It works as described, no complaints no praise.
2               Terrible experience, would not recommend.
3          Absolutely love this product, works perfectly!
4                  Decent for the price, could be better.
                              ...                        
3995    Great quality and fast delivery. Highly recomm...
3996       Absolutely love this product, works perfectly!
3997        Superb performance, exceeded my expectations.
3998        Good packaging and the product looks amazing.
3999           Excellent value for money, very satisfied.
Name: Review_Text, Length: 4000, dtype: str


In [12]:
#sentiment 
sent_map = {
    "positive": "Positive", "positve": "Positive", "pos": "Positive", "+ve": "Positive", "good": "Positive",
    "negative": "Negative", "negetive": "Negative", "neg": "Negative", "-ve": "Negative", "bad": "Negative",
    "neutral": "Neutral", "neutal": "Neutral", "neu": "Neutral", "average": "Neutral", "mixed": "Neutral",
}
df["Sentiment"] = df["Sentiment"].str.strip().str.lower().map(sent_map)
df["Rating"] = df["Rating"].fillna(df.groupby("Sentiment")["Rating"].transform("median")).astype(int)
print(df["Sentiment"])
print(df["Rating"])

0       Positive
1        Neutral
2       Negative
3       Positive
4        Neutral
          ...   
3995    Positive
3996    Positive
3997    Positive
3998    Positive
3999    Positive
Name: Sentiment, Length: 4000, dtype: str
0       5
1       3
2       2
3       5
4       3
       ..
3995    4
3996    5
3997    4
3998    4
3999    4
Name: Rating, Length: 4000, dtype: int64


In [13]:
#email
df["Email"] = (df["Email"].astype(str).str.replace(" ", "", regex=False)
               .str.replace("@@", "@", regex=False).str.lower())
email_ok = df["Email"].str.match(r"^[\w.+-]+@[\w-]+\.[a-z]{2,}$")
df.loc[~email_ok, "Email"] = "Unknown"
print(df["Email"])

0        ravisharma605@hotmail.com
1          aruniyer349@outlook.com
2        lakshmimenon722@gmail.com
3          meenadas168@hotmail.com
4         manojiyer253@outlook.com
                   ...            
3995       ajaysharma578@yahoo.com
3996     vijaymenon351@outlook.com
3997       manojmenon717@gmail.com
3998         manojdas392@yahoo.com
3999    lakshmipatel32@outlook.com
Name: Email, Length: 4000, dtype: str


In [14]:
#phone invalid length
df["Phone"] = df["Phone"].astype(str).str.replace(r"\D", "", regex=True)
df.loc[df["Phone"].str.len() != 10, "Phone"] = "Unknown"
print(df["Phone"])

0       6013389083
1       7351161559
2       6341316475
3       9483503056
4          Unknown
           ...    
3995    9976040271
3996    6035188934
3997    6616466632
3998    8584851662
3999    8168331421
Name: Phone, Length: 4000, dtype: str


In [15]:
#purchase amnt
df["Purchase_Amount"] = df["Purchase_Amount"].abs()
print(df["Purchase_Amount"])

0       14376.19
1        5436.51
2        3419.55
3        5500.27
4       17248.84
          ...   
3995    10750.74
3996    14994.06
3997     8821.20
3998    18739.21
3999    13454.16
Name: Purchase_Amount, Length: 4000, dtype: float64


In [16]:
#city
valid_cities = ["Coimbatore", "Delhi", "Mumbai", "Pune", "Chennai", "Hyderabad",
                "Jaipur", "Kochi", "Salem", "Madurai", "Kolkata", "Bangalore"]
df["City"] = df["City"].apply(
    lambda x: x.strip().title() if isinstance(x, str) and x.strip().title() in valid_cities else np.nan)
df["City"] = df["City"].fillna(df["City"].mode()[0])
df = df.reset_index(drop=True)
print(df["City"])
print("After:", df.shape)
print(df.isna().sum())
df.to_excel("Customer_Sentiment_Feedback_Cleaned.xlsx", index=False)

0            Kochi
1          Chennai
2          Kolkata
3        Bangalore
4            Delhi
           ...    
3995         Kochi
3996    Coimbatore
3997     Bangalore
3998          Pune
3999     Hyderabad
Name: City, Length: 4000, dtype: str
After: (4000, 14)
Customer_ID         0
Order_ID            0
Age                 0
Gender              0
Product_Name        0
Product_Category    0
Rating              0
Review_Date         0
Review_Text         0
Sentiment           0
Email               0
Phone               0
Purchase_Amount     0
City                0
dtype: int64
